# Modulare RAG-Pipeline

Austauschbare Komponenten:
- **Chunker**: zerlegt Dokumente in Passagen
- **Embedder**: erzeugt dichte Vektoren (für Dense-Retrieval)
- **Retriever**: Dense / BM25 / Hybrid (RRF)
- **Reranker**: ordnet die Top-100-Treffer um auf die finalen Top-10
- **Generator**: GPT-OSS-20B via LM Studio (OpenAI-kompatible API)

Pipeline: PDFs laden → chunken → indexieren → Retrieval@100 → Rerank@10 → Antwort generieren.

In [31]:
%pip install -q pypdf sentence-transformers scikit-learn numpy openai rank-bm25

Python(90978) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from __future__ import annotations

import math
import pathlib
import re
from abc import ABC, abstractmethod
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

PDF_DIR = pathlib.Path('pdfs')
TOP_K_RETRIEVE = 100
TOP_K_RERANK   = 10

## 1) PDFs laden

In [33]:
@dataclass
class Document:
    doc_id: str
    source: str
    text: str

def load_pdfs(pdf_dir: pathlib.Path) -> list[Document]:
    docs = []
    for pdf_path in sorted(pdf_dir.glob('*.pdf')):
        reader = PdfReader(str(pdf_path))
        pages = [p.extract_text() or '' for p in reader.pages]
        text = '\n\n'.join(pages)
        docs.append(Document(doc_id=pdf_path.stem, source=str(pdf_path), text=text))
    return docs

documents = load_pdfs(PDF_DIR)
print(f'{len(documents)} PDF(s) geladen.')
for d in documents[:3]:
    print(f'  - {d.doc_id}: {len(d.text)} Zeichen')

21 PDF(s) geladen.
  - beir_benchmark: 94693 Zeichen
  - bertscore: 136399 Zeichen
  - bleu: 27936 Zeichen


## 2) Chunker (austauschbar)

In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    text: str
    meta: dict = field(default_factory=dict)

class Chunker(ABC):
    @abstractmethod
    def chunk(self, doc: Document) -> list[Chunk]: ...

def _hard_split(s: str, max_chars: int) -> list[str]:
    """Schneidet einen einzelnen String in <= max_chars große Teile."""
    return [s[i:i + max_chars] for i in range(0, len(s), max_chars)] or [s]

class FixedSizeChunker(Chunker):
    """Zeichenbasierte Fenster mit Overlap. Schnell, sprachunabhängig."""
    def __init__(self, size: int = 800, overlap: int = 150):
        self.size, self.overlap = size, overlap

    def chunk(self, doc: Document) -> list[Chunk]:
        text = doc.text
        step = self.size - self.overlap
        chunks = []
        for i, start in enumerate(range(0, max(1, len(text)), step)):
            piece = text[start:start + self.size].strip()
            if piece:
                chunks.append(Chunk(f'{doc.doc_id}::c{i}', doc.doc_id, piece))
        return chunks

class SentenceChunker(Chunker):
    """Gruppiert ganze Sätze, bis ein Zeichenbudget erreicht ist.
    Übergroße 'Sätze' (z.B. PDFs ohne Satzpunkte) werden hart auf max_chars geschnitten."""
    _SPLIT = re.compile(r'(?<=[.!?])\s+')

    def __init__(self, max_chars: int = 800, overlap_sents: int = 1):
        self.max_chars, self.overlap_sents = max_chars, overlap_sents

    def chunk(self, doc: Document) -> list[Chunk]:
        raw = [s.strip() for s in self._SPLIT.split(doc.text) if s.strip()]
        sents: list[str] = []
        for s in raw:
            sents.extend(_hard_split(s, self.max_chars) if len(s) > self.max_chars else [s])

        chunks, buf, n, idx = [], [], 0, 0
        for s in sents:
            if n + len(s) > self.max_chars and buf:
                chunks.append(Chunk(f'{doc.doc_id}::c{idx}', doc.doc_id, ' '.join(buf)))
                idx += 1
                buf = buf[-self.overlap_sents:] if self.overlap_sents else []
                n = sum(len(x) for x in buf)
            buf.append(s); n += len(s)
        if buf:
            chunks.append(Chunk(f'{doc.doc_id}::c{idx}', doc.doc_id, ' '.join(buf)))
        return chunks

class ParagraphChunker(Chunker):
    """Splittet an Leerzeilen, packt kleine Absätze zusammen.
    Übergroße Absätze werden hart auf max_chars geschnitten."""
    def __init__(self, max_chars: int = 1000):
        self.max_chars = max_chars

    def chunk(self, doc: Document) -> list[Chunk]:
        raw = [p.strip() for p in re.split(r'\n\s*\n', doc.text) if p.strip()]
        paras: list[str] = []
        for p in raw:
            paras.extend(_hard_split(p, self.max_chars) if len(p) > self.max_chars else [p])

        chunks, buf, n, idx = [], [], 0, 0
        for p in paras:
            if n + len(p) > self.max_chars and buf:
                chunks.append(Chunk(f'{doc.doc_id}::c{idx}', doc.doc_id, '\n\n'.join(buf)))
                idx += 1; buf, n = [], 0
            buf.append(p); n += len(p)
        if buf:
            chunks.append(Chunk(f'{doc.doc_id}::c{idx}', doc.doc_id, '\n\n'.join(buf)))
        return chunks

## 3) Embedder (austauschbar)

In [ ]:
class Embedder(ABC):
    @abstractmethod
    def encode(self, texts: list[str], is_query: bool = False) -> np.ndarray: ...

class STEmbedder(Embedder):
    """sentence-transformers, normalisierte Vektoren."""
    def __init__(self, model_name: str = 'sentence-transformers/all-MiniLM-L6-v2', batch_size: int = 64):
        self.model = SentenceTransformer(model_name)
        self.batch_size = batch_size
        self.name = model_name

    def encode(self, texts: list[str], is_query: bool = False) -> np.ndarray:
        return self.model.encode(
            texts, batch_size=self.batch_size,
            normalize_embeddings=True, show_progress_bar=not is_query,
        )

class E5Embedder(Embedder):
    """E5-Familie verlangt 'query: ' / 'passage: ' Präfixe."""
    def __init__(self, model_name: str = 'intfloat/multilingual-e5-base', batch_size: int = 32):
        self.model = SentenceTransformer(model_name)
        self.batch_size = batch_size
        self.name = model_name

    def encode(self, texts: list[str], is_query: bool = False) -> np.ndarray:
        prefix = 'query: ' if is_query else 'passage: '
        prefixed = [prefix + t for t in texts]
        return self.model.encode(
            prefixed, batch_size=self.batch_size,
            normalize_embeddings=True, show_progress_bar=not is_query,
        )

class NomicEmbedder(Embedder):
    """Nomic Embed v1.5: 768-dim, Matryoshka-trunkierbar.
    Verlangt task-spezifische Präfixe ('search_query: ' / 'search_document: '),
    trust_remote_code=True und ein gesetztes max_seq_length — sonst sprengt
    die O(L²)-Attention bei langen Chunks den Speicher."""
    def __init__(
        self,
        model_name: str = 'nomic-ai/nomic-embed-text-v1.5',
        batch_size: int = 16,
        max_seq_length: int = 512,
        matryoshka_dim: int | None = None,  # 64..768
    ):
        self.model = SentenceTransformer(model_name, trust_remote_code=True)
        self.model.max_seq_length = max_seq_length
        self.batch_size = batch_size
        self.matryoshka_dim = matryoshka_dim
        self.name = model_name + (f'@{matryoshka_dim}d' if matryoshka_dim else '')

    def encode(self, texts: list[str], is_query: bool = False) -> np.ndarray:
        prefix = 'search_query: ' if is_query else 'search_document: '
        prefixed = [prefix + t for t in texts]
        emb = self.model.encode(
            prefixed, batch_size=self.batch_size,
            normalize_embeddings=False, show_progress_bar=not is_query,
        )
        if self.matryoshka_dim:
            emb = emb[:, :self.matryoshka_dim]
        norms = np.linalg.norm(emb, axis=1, keepdims=True)
        return emb / np.clip(norms, 1e-12, None)

## 4) Reranker (austauschbar)

In [36]:
class Reranker(ABC):
    @abstractmethod
    def rerank(self, query: str, candidates: list[tuple[Chunk, float]], top_k: int) -> list[tuple[Chunk, float]]: ...

class NoOpReranker(Reranker):
    """Behält die Reihenfolge des Retrievers."""
    def rerank(self, query, candidates, top_k):
        return candidates[:top_k]

class CrossEncoderReranker(Reranker):
    """Cross-Encoder bewertet (query, passage) gemeinsam."""
    def __init__(self, model_name: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2', batch_size: int = 32):
        self.model = CrossEncoder(model_name)
        self.batch_size = batch_size
        self.name = model_name

    def rerank(self, query, candidates, top_k):
        pairs = [(query, c.text) for c, _ in candidates]
        scores = self.model.predict(pairs, batch_size=self.batch_size, show_progress_bar=False)
        scored = list(zip([c for c, _ in candidates], map(float, scores)))
        scored.sort(key=lambda x: -x[1])
        return scored[:top_k]

## 5) Retriever (austauschbar): Dense, BM25, Hybrid

- `DenseRetriever`: Cosine auf Embeddings (braucht einen `Embedder`).
- `BM25Retriever`: lexikalisch, kein Modell nötig.
- `HybridRetriever`: kombiniert zwei Retriever via **Reciprocal Rank Fusion** (`score = Σ 1/(k+rank)`).

In [37]:
def tokenize(text: str) -> list[str]:
    return re.findall(r'[a-z0-9]+', text.lower())

class Retriever(ABC):
    @abstractmethod
    def build(self, chunks: list[Chunk]) -> None: ...
    @abstractmethod
    def search(self, query: str, top_k: int) -> list[tuple[Chunk, float]]: ...

class DenseRetriever(Retriever):
    """Cosine via Skalarprodukt auf normalisierten Embeddings."""
    def __init__(self, embedder: Embedder):
        self.embedder = embedder
        self.chunks: list[Chunk] = []
        self.matrix: np.ndarray | None = None
        self.name = f'Dense({getattr(embedder, "name", type(embedder).__name__)})'

    def build(self, chunks):
        self.chunks = chunks
        self.matrix = self.embedder.encode([c.text for c in chunks], is_query=False)

    def search(self, query, top_k):
        q = self.embedder.encode([query], is_query=True)
        sims = cosine_similarity(q, self.matrix)[0]
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.chunks[i], float(sims[i])) for i in top_idx]

class BM25Retriever(Retriever):
    """Lexikalischer Retriever (rank-bm25, Okapi)."""
    def __init__(self):
        self.chunks: list[Chunk] = []
        self.bm25: BM25Okapi | None = None
        self.name = 'BM25'

    def build(self, chunks):
        self.chunks = chunks
        self.bm25 = BM25Okapi([tokenize(c.text) for c in chunks])

    def search(self, query, top_k):
        scores = self.bm25.get_scores(tokenize(query))
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in top_idx]

class HybridRetriever(Retriever):
    """Reciprocal Rank Fusion über mehrere Retriever.

    rrf_score(c) = Σ_r 1 / (k + rank_r(c))
    Fragt jeden Sub-Retriever mit `fetch_k` (>> top_k) ab und fusioniert.
    """
    def __init__(self, retrievers: list[Retriever], k: int = 60, fetch_k: int = 200):
        self.retrievers = retrievers
        self.k = k
        self.fetch_k = fetch_k
        self.name = 'Hybrid(' + '+'.join(r.name for r in retrievers) + ')'

    def build(self, chunks):
        for r in self.retrievers:
            r.build(chunks)

    def search(self, query, top_k):
        fused: dict[str, float] = {}
        chunk_lookup: dict[str, Chunk] = {}
        for r in self.retrievers:
            for rank, (chunk, _) in enumerate(r.search(query, self.fetch_k)):
                fused[chunk.chunk_id] = fused.get(chunk.chunk_id, 0.0) + 1.0 / (self.k + rank + 1)
                chunk_lookup[chunk.chunk_id] = chunk
        ranked = sorted(fused.items(), key=lambda kv: -kv[1])[:top_k]
        return [(chunk_lookup[cid], score) for cid, score in ranked]

## 6) Generator (LM Studio, OpenAI-kompatibel)

LM Studio läuft lokal unter `http://localhost:1234/v1`. Modell: `gpt-oss-20b` (so wie es in LM Studio geladen ist).

In [38]:
class Generator(ABC):
    @abstractmethod
    def generate(self, query: str, context_chunks: list[Chunk]) -> str: ...

SYSTEM_PROMPT = (
    'Du bist ein präziser Assistent. Beantworte die Frage AUSSCHLIESSLICH auf Basis '
    'der bereitgestellten Quellen. Zitiere die verwendeten Quellen-IDs in eckigen '
    'Klammern, z. B. [doc::c3]. Wenn die Quellen die Frage nicht beantworten, '
    'sage das explizit.'
)

def fit_context(chunks: list[Chunk], char_budget: int) -> list[Chunk]:
    """Greift in Reihenfolge zu, bis das Zeichenbudget erreicht ist.
    Letzter passender Chunk wird ggf. abgeschnitten."""
    out, used = [], 0
    for c in chunks:
        framing = len(c.chunk_id) + 4  # für '[id] '
        remaining = char_budget - used - framing
        if remaining <= 100:
            break
        if len(c.text) <= remaining:
            out.append(c); used += framing + len(c.text)
        else:
            out.append(Chunk(c.chunk_id, c.doc_id, c.text[:remaining] + '…'))
            break
    return out

def format_context(chunks: list[Chunk]) -> str:
    return '\n\n'.join(f'[{c.chunk_id}] {c.text}' for c in chunks)

class LMStudioGenerator(Generator):
    def __init__(
        self,
        model: str = 'gpt-oss-20b',
        base_url: str = 'http://localhost:1234/v1',
        api_key: str = 'lm-studio',
        temperature: float = 0.2,
        max_tokens: int = 800,
        context_char_budget: int = 6000,  # ~1.5–2k Tokens; passt zu kleinem LM-Studio-Context
    ):
        self.client = OpenAI(base_url=base_url, api_key=api_key)
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.context_char_budget = context_char_budget

    def generate(self, query: str, context_chunks: list[Chunk]) -> str:
        fitted = fit_context(context_chunks, self.context_char_budget)
        user_msg = (
            f'Frage: {query}\n\n'
            f'Quellen:\n{format_context(fitted)}\n\n'
            'Antwort:'
        )
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': user_msg},
            ],
            temperature=self.temperature,
            max_tokens=self.max_tokens,
        )
        return resp.choices[0].message.content

## 7) Pipeline

In [39]:
@dataclass
class RAGResult:
    query: str
    answer: str
    retrieved: list[tuple[Chunk, float]]
    reranked: list[tuple[Chunk, float]]

class RAGPipeline:
    def __init__(
        self,
        chunker: Chunker,
        retriever: Retriever,
        reranker: Reranker,
        generator: Generator,
        top_k_retrieve: int = TOP_K_RETRIEVE,
        top_k_rerank: int = TOP_K_RERANK,
    ):
        self.chunker = chunker
        self.retriever = retriever
        self.reranker = reranker
        self.generator = generator
        self.top_k_retrieve = top_k_retrieve
        self.top_k_rerank = top_k_rerank

    def ingest(self, documents: list[Document]) -> None:
        all_chunks: list[Chunk] = []
        for d in documents:
            all_chunks.extend(self.chunker.chunk(d))
        print(f'{len(all_chunks)} Chunks aus {len(documents)} Dokument(en). Retriever: {self.retriever.name}')
        self.retriever.build(all_chunks)

    def ask(self, query: str) -> RAGResult:
        retrieved = self.retriever.search(query, self.top_k_retrieve)
        reranked  = self.reranker.rerank(query, retrieved, self.top_k_rerank)
        answer    = self.generator.generate(query, [c for c, _ in reranked])
        return RAGResult(query=query, answer=answer, retrieved=retrieved, reranked=reranked)

## 8) Konfiguration & Demo

Einzelne Komponenten austauschen, indem man hier eine andere Klasse instanziiert.

In [ ]:
pipeline = RAGPipeline(
    chunker  = SentenceChunker(max_chars=800, overlap_sents=1),
    retriever = HybridRetriever(
        retrievers=[
            BM25Retriever(),
            DenseRetriever(STEmbedder('sentence-transformers/all-MiniLM-L6-v2')),
        ],
        k=60, fetch_k=200,
    ),
    reranker = CrossEncoderReranker('cross-encoder/ms-marco-MiniLM-L-6-v2'),
    generator = LMStudioGenerator(model='gpt-oss-20b'),
    top_k_retrieve=TOP_K_RETRIEVE,
    top_k_rerank=TOP_K_RERANK,
)
pipeline.ingest(documents)

In [94]:
#result = pipeline.ask('Worum geht es in den Dokumenten?')

result = pipeline.ask('Welche Metriken sind wichtig für den Reranker? Antworte in Stichpunkten') 

print('=== Antwort ===')
print(result.answer)
print('\n=== Top-10 (rerankt) ===')
for rank, (c, score) in enumerate(result.reranked, 1):
    preview = c.text[:120].replace('\n', ' ')
    print(f'{rank:>2}. [{score:+.3f}] {c.chunk_id}  —  {preview}…')

=== Antwort ===


=== Top-10 (rerankt) ===
 1. [-8.393] truthfulqa::c79  —  Francis Song, John Aslanides, Sarah Henderson, Roman Ring, Susan- nah Young, Eliza Rutherford, Tom Hennigan, Ja- cob Men…
 2. [-8.485] mteb_benchmark::c60  —  Reranking MPNet and MiniLM models perform strongly on reranking tasks. On SciDocsRR (Co- han et al., 2020a) they perform…
 3. [-9.697] rag_best_practices::c122  —  All reranking methods demonstrate a notable 9https://huggingface.co/HuggingFaceH4/zephyr-7b-alpha 19  Context Model NQ T…
 4. [-10.082] mteb_benchmark::c107  —   StackExchangeClusteringP2P Clustering p2p 1 0 0 75000 0 0 1090.7 TwentyNewsgroupsClustering Clustering s2s 1 0 0 59545 …
 5. [-10.179] truthfulqa::c78  —  Rae, Sebastian Borgeaud, Trevor Cai, Katie Millican, Jordan Hoffmann, H. Francis Song, John Aslanides, Sarah Henderson, …
 6. [-10.292] mteb_benchmark::c24  —  ivClusteringP2P ArxivClusteringS2S BiorxivClusteringP2P BiorxivClusteringS2S MedrxivClusteringP2P MedrxivClusteringS2S R…
 7

### Variante: BM25 only (kein Embedder, kein Reranker)

In [99]:
pipeline_bm25 = RAGPipeline(
    chunker  = ParagraphChunker(max_chars=1000),
    retriever = BM25Retriever(),
    reranker = NoOpReranker(),
    generator = LMStudioGenerator(model='gpt-oss-20b'),
)
pipeline_bm25.ingest(documents)
# result = pipeline_bm25.ask('...')

1757 Chunks aus 21 Dokument(en). Retriever: BM25


### Variante: Hybrid (BM25 + Dense) mit Cross-Encoder-Reranker

In [105]:
pipeline_hybrid = RAGPipeline(
    chunker  = SentenceChunker(max_chars=800, overlap_sents=1),
    retriever = HybridRetriever(
        retrievers=[
            BM25Retriever(),
            DenseRetriever(E5Embedder('intfloat/multilingual-e5-base')),
        ],
        k=60, fetch_k=200,
    ),
    reranker = CrossEncoderReranker('cross-encoder/ms-marco-MiniLM-L-6-v2'),
    generator = LMStudioGenerator(model='gpt-oss-20b'),
)
pipeline_hybrid.ingest(documents)
# result = pipeline_hybrid.ask('...')


2635 Chunks aus 21 Dokument(en). Retriever: Hybrid(BM25+Dense(intfloat/multilingual-e5-base))


Batches: 100%|██████████| 83/83 [00:25<00:00,  3.29it/s]


## 9) Gold-Set & Retrieval-Metriken

50 Fragen mit **graded relevance 0…4** auf **Dokumenten-Ebene** (`doc_id` = PDF-Stem).
Begründung: Chunk-IDs ändern sich mit jedem Chunker; Doc-Labels sind stabil.

Skala:
- **4** = Kernpaper / definiert das Thema
- **3** = stark relevant / behandelt das Thema ausführlich
- **2** = relevant / sekundärer Bezug
- **1** = randständig erwähnt
- **0** = nicht relevant (nicht aufgeführt)

Chunk-Treffer werden per **max-score** auf Dokument-Ebene aggregiert, dann gegen das Gold-Set bewertet.

In [110]:
QRELS: dict[str, dict] = {
    'q01': {'query': 'Was ist BLEU und wie wird es berechnet?',
            'rels': {'bleu': 4, 'rouge': 2, 'bertscore': 2, 'geval': 1}},
    'q02': {'query': 'Wie funktioniert ROUGE als Metrik?',
            'rels': {'rouge': 4, 'bleu': 2, 'bertscore': 2}},
    'q03': {'query': 'Was misst BERTScore und wozu dient es?',
            'rels': {'bertscore': 4, 'bleu': 2, 'rouge': 2, 'geval': 2}},
    'q04': {'query': 'Wie funktioniert G-Eval mit LLMs als Evaluator?',
            'rels': {'geval': 4, 'human_evaluation_tutorial': 2, 'ragbench': 1}},
    'q05': {'query': 'Was ist HNSW (Hierarchical Navigable Small World)?',
            'rels': {'hnsw': 4, 'dense_passage_retrieval': 1}},
    'q06': {'query': 'Wie funktioniert Dense Passage Retrieval?',
            'rels': {'dense_passage_retrieval': 4, 'sentence_bert': 2, 'beir_benchmark': 1}},
    'q07': {'query': 'Was ist Sentence-BERT und wozu wird es benutzt?',
            'rels': {'sentence_bert': 4, 'dense_passage_retrieval': 2}},
    'q08': {'query': 'Was ist der BEIR Benchmark?',
            'rels': {'beir_benchmark': 4, 'mteb_benchmark': 2, 'ragbench': 1}},
    'q09': {'query': 'Was ist MTEB (Massive Text Embedding Benchmark)?',
            'rels': {'mteb_benchmark': 4, 'beir_benchmark': 2, 'sentence_bert': 1}},
    'q10': {'query': 'Was ist der CRAG Benchmark für RAG-Systeme?',
            'rels': {'crag_benchmark': 4, 'ragbench': 2, 'rag_best_practices': 1}},
    'q11': {'query': 'Was bewertet RAGBench?',
            'rels': {'ragbench': 4, 'crag_benchmark': 2, 'rag_best_practices': 1}},
    'q12': {'query': 'Wie misst man Halluzinationen in Sprachmodellen?',
            'rels': {'halueval': 4, 'truthfulqa': 3, 'bold': 2, 'geval': 1}},
    'q13': {'query': 'Was ist HaluEval?',
            'rels': {'halueval': 4, 'truthfulqa': 2}},
    'q14': {'query': 'Was ist TruthfulQA?',
            'rels': {'truthfulqa': 4, 'halueval': 2}},
    'q15': {'query': 'Was misst der BOLD Bias-Benchmark?',
            'rels': {'bold': 4, 'truthfulqa': 1}},
    'q16': {'query': 'Was bedeutet "Lost in the Middle" bei langen Kontexten?',
            'rels': {'lost_in_the_middle': 4, 'rag_best_practices': 2}},
    'q17': {'query': 'Was ist TextTiling für Textsegmentierung?',
            'rels': {'texttiling': 4}},
    'q18': {'query': 'Wie chunkt man Dokumente sinnvoll für RAG?',
            'rels': {'texttiling': 4, 'rag_best_practices': 3, 'sentence_bert': 1}},
    'q19': {'query': 'Was ist DriftLens?',
            'rels': {'driftlens': 4}},
    'q20': {'query': 'Wie erkennt man semantischen Drift in Embeddings?',
            'rels': {'driftlens': 4, 'rag_best_practices': 1}},
    'q21': {'query': 'Wie evaluiert man Information-Retrieval-Systeme?',
            'rels': {'stanford_ir_eval_ch8': 4, 'beir_benchmark': 3, 'mteb_benchmark': 2}},
    'q22': {'query': 'Was ist nDCG (normalized Discounted Cumulative Gain)?',
            'rels': {'stanford_ir_eval_ch8': 4, 'beir_benchmark': 2}},
    'q23': {'query': 'Wie berechnet man Mean Average Precision (MAP)?',
            'rels': {'stanford_ir_eval_ch8': 4, 'beir_benchmark': 2}},
    'q24': {'query': 'Was ist Mean Reciprocal Rank (MRR)?',
            'rels': {'stanford_ir_eval_ch8': 4}},
    'q25': {'query': 'Was bedeuten Precision und Recall in IR?',
            'rels': {'stanford_ir_eval_ch8': 4}},
    'q26': {'query': 'Wie führt man eine Human Evaluation für NLP-Systeme durch?',
            'rels': {'human_evaluation_tutorial': 4, 'geval': 1}},
    'q27': {'query': 'Was sind Best Practices für RAG-Pipelines?',
            'rels': {'rag_best_practices': 4, 'rag_paper': 3, 'lost_in_the_middle': 2}},
    'q28': {'query': 'Was ist Retrieval-Augmented Generation (RAG)?',
            'rels': {'rag_paper': 4, 'rag_best_practices': 3, 'dense_passage_retrieval': 2}},
    'q29': {'query': 'Wozu braucht man einen Reranker im RAG-Pipeline?',
            'rels': {'rag_best_practices': 3, 'rag_paper': 2, 'dense_passage_retrieval': 1}},
    'q30': {'query': 'Was ist hybrid retrieval (BM25 + Dense)?',
            'rels': {'rag_best_practices': 3, 'beir_benchmark': 1, 'dense_passage_retrieval': 1}},
    'q31': {'query': 'Was ist Approximate Nearest Neighbor Search?',
            'rels': {'hnsw': 4}},
    'q32': {'query': 'Wie verbessert HNSW die Vektorsuche im Vergleich zu Brute-Force?',
            'rels': {'hnsw': 4}},
    'q33': {'query': 'Wie unterscheiden sich Bi-Encoder und Cross-Encoder?',
            'rels': {'sentence_bert': 3, 'dense_passage_retrieval': 2}},
    'q34': {'query': 'Wie misst man Bias und Fairness in LLM-Outputs?',
            'rels': {'bold': 4, 'truthfulqa': 3, 'halueval': 1}},
    'q35': {'query': 'Was bedeutet Faithfulness bei RAG-generierten Antworten?',
            'rels': {'ragbench': 3, 'rag_best_practices': 3, 'halueval': 2, 'geval': 2}},
    'q36': {'query': 'Was ist Context Compression bei langen Prompts?',
            'rels': {'lost_in_the_middle': 3, 'rag_best_practices': 2}},
    'q37': {'query': 'Welches Embedding-Modell sollte ich für meine Domain wählen?',
            'rels': {'mteb_benchmark': 4, 'sentence_bert': 3, 'dense_passage_retrieval': 2}},
    'q38': {'query': 'Wie groß sollte ein Chunk in einem RAG-System sein?',
            'rels': {'rag_best_practices': 3, 'texttiling': 2, 'lost_in_the_middle': 2}},
    'q39': {'query': 'Was ist semantisches Chunking?',
            'rels': {'texttiling': 4, 'rag_best_practices': 2}},
    'q40': {'query': 'Wie evaluiert man die Antwortqualität von RAG-Systemen?',
            'rels': {'ragbench': 4, 'crag_benchmark': 3, 'geval': 3, 'rag_best_practices': 2,
                     'human_evaluation_tutorial': 2}},
    'q41': {'query': 'Was sind faktische Halluzinationen bei Generator-LLMs?',
            'rels': {'halueval': 4, 'truthfulqa': 3, 'geval': 2}},
    'q42': {'query': 'Was ist DCG (Discounted Cumulative Gain)?',
            'rels': {'stanford_ir_eval_ch8': 4}},
    'q43': {'query': 'Wie wirkt sich die Position relevanter Information im Kontextfenster aus?',
            'rels': {'lost_in_the_middle': 4, 'rag_best_practices': 1}},
    'q44': {'query': 'Welche Datasets gibt es zur Evaluierung von Retrieval?',
            'rels': {'beir_benchmark': 4, 'mteb_benchmark': 3, 'stanford_ir_eval_ch8': 1}},
    'q45': {'query': 'Wie funktioniert N-Gram-basierte Bewertung von Übersetzungen?',
            'rels': {'bleu': 4, 'rouge': 4}},
    'q46': {'query': 'Was sind contextual word embeddings?',
            'rels': {'sentence_bert': 3, 'bertscore': 3, 'dense_passage_retrieval': 2}},
    'q47': {'query': 'Wie sensitiv sind LLM-as-Judge-Bewertungen gegenüber Prompts?',
            'rels': {'geval': 3, 'human_evaluation_tutorial': 2}},
    'q48': {'query': 'Was ist tf-idf?',
            'rels': {'stanford_ir_eval_ch8': 3}},
    'q49': {'query': 'Wie funktioniert BM25?',
            'rels': {'stanford_ir_eval_ch8': 3, 'beir_benchmark': 2, 'dense_passage_retrieval': 2}},
    'q50': {'query': 'Wie skalieren Vektorindizes bei Millionen von Dokumenten?',
            'rels': {'hnsw': 4, 'dense_passage_retrieval': 1}},
}

# Sanity-Checks: alle referenzierten doc_ids existieren als geladene PDFs
known = {d.doc_id for d in documents}
for qid, q in QRELS.items():
    unknown = set(q['rels']) - known
    if unknown:
        print(f'WARN {qid}: unbekannte doc_ids {unknown}')
print(f'Gold-Set: {len(QRELS)} Queries, {sum(len(q["rels"]) for q in QRELS.values())} Labels gesamt.')

Gold-Set: 50 Queries, 123 Labels gesamt.


In [107]:
def chunks_to_doc_ranking(chunk_results: list[tuple[Chunk, float]]) -> list[str]:
    """Aggregiert Chunk-Treffer auf Dokument-Ebene (max-score pro doc_id)."""
    best: dict[str, float] = {}
    for c, s in chunk_results:
        if c.doc_id not in best or s > best[c.doc_id]:
            best[c.doc_id] = s
    return [d for d, _ in sorted(best.items(), key=lambda kv: -kv[1])]

def precision_at_k(ranked: list[str], rels: dict[str, int], k: int) -> float:
    return sum(1 for d in ranked[:k] if rels.get(d, 0) >= 1) / k

def recall_at_k(ranked: list[str], rels: dict[str, int], k: int) -> float:
    total = sum(1 for r in rels.values() if r >= 1)
    if total == 0:
        return 0.0
    return sum(1 for d in ranked[:k] if rels.get(d, 0) >= 1) / total

def _dcg(grades: list[int], k: int) -> float:
    return sum((2**g - 1) / math.log2(i + 2) for i, g in enumerate(grades[:k]))

def ndcg_at_k(ranked: list[str], rels: dict[str, int], k: int) -> float:
    """Graded nDCG: nutzt rel ∈ {0..4} direkt im DCG-Gewicht (2^rel - 1)."""
    grades = [rels.get(d, 0) for d in ranked[:k]]
    ideal  = sorted(rels.values(), reverse=True)
    idcg = _dcg(ideal, k)
    return _dcg(grades, k) / idcg if idcg > 0 else 0.0

def average_precision(ranked: list[str], rels: dict[str, int], cap: int = 100) -> float:
    total = sum(1 for r in rels.values() if r >= 1)
    if total == 0:
        return 0.0
    hits, ap = 0, 0.0
    for i, d in enumerate(ranked[:cap], start=1):
        if rels.get(d, 0) >= 1:
            hits += 1
            ap += hits / i
    return ap / total

def reciprocal_rank(ranked: list[str], rels: dict[str, int]) -> float:
    for i, d in enumerate(ranked, start=1):
        if rels.get(d, 0) >= 1:
            return 1.0 / i
    return 0.0

def evaluate_retriever(retriever: Retriever, qrels: dict, top_k: int = TOP_K_RETRIEVE) -> dict[str, float]:
    rows = {'P@5': [], 'R@10': [], 'nDCG@10': [], 'MAP@100': [], 'MRR': []}
    for q in qrels.values():
        results = retriever.search(q['query'], top_k)
        ranked = chunks_to_doc_ranking(results)
        rels = q['rels']
        rows['P@5'].append(precision_at_k(ranked, rels, 5))
        rows['R@10'].append(recall_at_k(ranked, rels, 10))
        rows['nDCG@10'].append(ndcg_at_k(ranked, rels, 10))
        rows['MAP@100'].append(average_precision(ranked, rels, 100))
        rows['MRR'].append(reciprocal_rank(ranked, rels))
    return {m: float(np.mean(v)) for m, v in rows.items()}

### Vergleich: BM25 vs. Dense vs. Hybrid

Der Hybrid-Retriever (`pipeline.retriever`) hat seine Sub-Retriever (BM25, Dense) bereits gebaut — wir greifen direkt darauf zu, ohne neu zu indexieren.

In [109]:
bm25_r, dense_r = pipeline.retriever.retrievers
hybrid_r = pipeline.retriever

results = {
    'BM25':         evaluate_retriever(bm25_r,   QRELS),
    'Dense':        evaluate_retriever(dense_r,  QRELS),
    'Hybrid (RRF)': evaluate_retriever(hybrid_r, QRELS),
}

df = pd.DataFrame(results).round(3)
df['Δ Hybrid−Dense'] = (df['Hybrid (RRF)'] - df['Dense']).round(3)
df['Δ Hybrid−BM25']  = (df['Hybrid (RRF)'] - df['BM25']).round(3)
df

,BM25,Dense,Hybrid (RRF),Δ Hybrid−Dense,Δ Hybrid−BM25
P@5,0.292,0.348,0.344,-0.004,0.052
R@10,0.748,0.832,0.839,0.007,0.091
nDCG@10,0.669,0.780,0.798,0.018,0.129
MAP@100,0.549,0.660,0.663,0.003,0.114
MRR,0.726,0.853,0.874,0.021,0.148


### Per-Query-Drilldown

Welche Queries laufen schlecht? Hilft beim Schärfen der Pipeline (Chunker, Embedder, Reranker).

In [104]:
def per_query_ndcg(retriever: Retriever, qrels: dict, k: int = 10, top_k: int = TOP_K_RETRIEVE) -> pd.DataFrame:
    rows = []
    for qid, q in qrels.items():
        ranked = chunks_to_doc_ranking(retriever.search(q['query'], top_k))
        rows.append({'qid': qid, 'query': q['query'][:60],
                     'nDCG@10': round(ndcg_at_k(ranked, q['rels'], k), 3),
                     'top1': ranked[0] if ranked else None,
                     'top1_rel': q['rels'].get(ranked[0], 0) if ranked else 0})
    return pd.DataFrame(rows)

per_q = per_query_ndcg(hybrid_r, QRELS)
print('Schwächste 10 Queries (Hybrid):')
per_q.nsmallest(10, 'nDCG@10').to_string(index=False)

Schwächste 10 Queries (Hybrid):


'qid                                                        query  nDCG@10                    top1  top1_rel\nq48                                              Was ist tf-idf?    0.000                    hnsw         0\nq37 Welches Embedding-Modell sollte ich für meine Domain wählen?    0.352               bertscore         0\nq24                          Was ist Mean Reciprocal Rank (MRR)?    0.387               bertscore         0\nq50    Wie skalieren Vektorindizes bei Millionen von Dokumenten?    0.413               driftlens         0\nq39                               Was ist semantisches Chunking?    0.434      rag_best_practices         2\nq11                                       Was bewertet RAGBench?    0.458          crag_benchmark         2\nq47 Wie sensitiv sind LLM-as-Judge-Bewertungen gegenüber Prompts    0.470          crag_benchmark         0\nq49                                       Wie funktioniert BM25?    0.471 dense_passage_retrieval         2\nq18               